# This notebook shows how to scale the spin-orbit coupling term independently per chemical species
### Import the necessary libraries

In [1]:
from elkpy.structure import Structure

### The spin-orbit term, and Elk's single global scale
Elk adds the Koelling-Harmon spin-orbit term inside each muffin-tin sphere,
$$ \hat H_{\rm soc}(r)=f_{\rm soc}(r)\,\hat{\mathbf L}\cdot\boldsymbol\sigma, \qquad
f_{\rm soc}(r)=\frac1{(2M(r)c)^2}\,\frac1r\,\frac{\partial V_s(r)}{\partial r} $$
multiplied by one global scale, `socscf`, the same number for every atom regardless of species. elkpy promotes this to a per-species scale $s_{is}$,
$$ \texttt{socscf}\to s_{is}=\begin{cases}\texttt{soc\_scale[is]}&\text{if given}\\ \texttt{socscf}&\text{otherwise}\end{cases} $$
so a correction fitted for one (typically heavy) species doesn't leak onto a lighter one sharing the cell.

### A per-species override reproduces the global scale, for a single-species cell

In [2]:
BI_AVEC = [(0.0, 5.0, 5.0), (5.0, 0.0, 5.0), (5.0, 5.0, 0.0)]
bi = Structure(BI_AVEC, {"Bi": [(0.0, 0.0, 0.0)]})

e_global = bi.get_calculation(
    "_scratch/bi_global", xc="PW", spinorb=True, ngridk=(1, 1, 1), extra_blocks={"socscf": [3.0]}
).get_energy()
e_per_species = bi.get_calculation(
    "_scratch/bi_per_species", xc="PW", spinorb=True, ngridk=(1, 1, 1), soc_scale={"Bi": 3.0}
).get_energy()
print(e_global, e_per_species)

-21569.740146956 -21569.740146956


### Independent scaling in a two-species cell
Bismuth ($Z=83$, strong SOC) and silicon ($Z=14$, weak SOC): turning off each species' $s_{is}$ in turn should move $E$ away from the `spinorb=True` default, with the Bi effect dominating -- $f_{\rm soc}(r)$ grows strongly with $Z$ (the near-nuclear physics behind atomic fine structure).

In [3]:
BISI_AVEC = [(10.0, 0.0, 0.0), (0.0, 10.0, 0.0), (0.0, 0.0, 10.0)]
bisi = Structure(BISI_AVEC, {"Bi": [(0.0, 0.0, 0.0)], "Si": [(0.5, 0.5, 0.5)]})

def energy(label, soc_scale):
    return bisi.get_calculation(
        f"_scratch/{label}", xc="PW", spinorb=True, ngridk=(1, 1, 1), soc_scale=soc_scale
    ).get_energy()

e0, e_bi_off, e_si_off = energy("bisi0", None), energy("bisi_bi0", {"Bi": 0.0}), energy("bisi_si0", {"Si": 0.0})
print(abs(e0 - e_bi_off), abs(e0 - e_si_off))

0.0312532360003388 0.0005423130023700651
